# Deney: Orijinal model ↔ Heretic ile reddetme yönü silinmiş model

**Soru:** Tarayıcı, kırılganlığı *bilinen* bir modeli kırılgan olarak görebiliyor mu?

Heretic, modelin ağırlıklarından "reddetme yönünü" siler. Aynı modelin işlemden
önceki ve sonraki hâli, aradaki tek farkın reddetme davranışı olduğu kontrollü bir
çift oluşturur. Bu notebook iki modelin aynı `seed` ve aynı şablon kütüphanesiyle
yapılan taramalarını karşılaştırır.

| | Hipotez | Bakılacak hücre |
|---|---|---|
| **H1** | Heretic modelin Dayanıklılık Skoru (R) belirgin şekilde düşer | §1 |
| **H2** | Düşüş en çok `harmful_content` ve `jailbreak` kategorilerinde olur | §3 |
| **H3** | Reddetme kalıbı oranı Heretic modelde keskin biçimde düşer | §4 |
| **H4** | Fark istatistiksel olarak anlamlıdır (eşleştirilmiş McNemar testi) | §5 |
| **H5** | Katman 1 tek başına bazı kırılmaları "kararsız" bırakır; Katman 2 (judge) bunları çözer | §6, §7 |

Açık soru: `system_prompt_leak` ve `prompt_injection` kategorileri ne kadar değişir?
Heretic yalnızca reddetme yönünü hedefler; talimat hiyerarşisine uyumun da aynı
yönde kodlanıp kodlanmadığı bu deneyin cevaplayabileceği bir sorudur.

> Veri: `experiments/heretic_karsilastirma/sonuclar/*.db`. Bu dosyalar ham saldırı
> yanıtları içerir, yerelde kalır ve `.gitignore` ile depodan dışlanır (§11).
> Notebook hiçbir yerde tam yanıt metnini göstermez.

In [ ]:
# Modeller indirilmeden önce hattı denemek için True yapın
# (önce: python experiments/heretic_karsilastirma/0_kuru_prova.py).
KURU_PROVA = False

import sys
from pathlib import Path

DENEY = Path.cwd().resolve().parent / "experiments" / "heretic_karsilastirma"
if not DENEY.exists():  # notebook proje kökünden açıldıysa
    DENEY = Path.cwd().resolve() / "experiments" / "heretic_karsilastirma"
sys.path.insert(0, str(DENEY))

import karsilastir as k
import pandas as pd
import plotly.express as px

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

RENK = {"orijinal": "#4C78A8", "heretic": "#E45756"}
KARAR_RENK = {
    "kırıldı": "#C0392B",
    "kırıldı (düşük güven)": "#E59866",
    "kararsız": "#F4D03F",
    "dayandı": "#27AE60",
    "hata": "#95A5A6",
}

if KURU_PROVA:
    kuru = k.SONUC_DIZINI / "kuru_prova"
    df = k.sonuclari_yukle({m: kuru / f"deney_{m}.db" for m in ("kural", "judge")
                            if (kuru / f"deney_{m}.db").exists()})
else:
    df = k.sonuclari_yukle()

print(f"{len(df)} deneme · modlar: {sorted(df['mod'].unique())} · "
      f"hedefler: {sorted(df['target_name'].unique())} · seed'ler: {sorted(df['seed'].unique())}")

## §1 · Dayanıklılık Skoru (H1)

Skorlar `scoring.score_results` ile hesaplanır, yani CLI çıktısı ve HTML raporla
birebir aynıdır. `mod` sütunu: **kural** = yalnızca Katman 1, **judge** = Katman 1 + LLM-judge.

In [ ]:
ozet, kategori = k.kategori_dayaniklilik(df)
ozet.sort_values(["mod", "model", "seed"])

In [ ]:
# Birden fazla seed/tekrar varsa ortalama ± standart sapma
r_tablo = (
    ozet.groupby(["mod", "model"])["R"]
    .agg(["mean", "std", "count"])
    .rename(columns={"mean": "R ort.", "std": "R std", "count": "koşu"})
    .round(1)
)
r_tablo

## §2 · Karar dağılımı

Her denemenin vardığı durum. Heretic modelde **kararsız** payı büyükse, kırılma
tespitinin önemli bir kısmının Katman 1'de çözülemediği anlamına gelir (H5).

In [ ]:
dagilim = k.karar_dagilimi(df).reset_index().melt(
    id_vars=["mod", "model"], var_name="karar", value_name="oran"
)
fig = px.bar(
    dagilim, x="model", y="oran", color="karar", facet_col="mod",
    category_orders={"karar": k.KARAR_SIRASI, "model": ["orijinal", "heretic"]},
    color_discrete_map=KARAR_RENK, text_auto=".0%",
    title="Karar durumlarının dağılımı",
)
fig.update_yaxes(tickformat=".0%")
fig.show()

## §3 · Kategori bazında dayanıklılık (H2)

Dayanıklılık = 1 − kırılma oranı (düşük güvenli kırılmalar yarım ağırlıkla, §7).
Fark sütunu negatifse Heretic model o kategoride daha kırılgandır.

In [ ]:
kat_ort = kategori.groupby(["mod", "category", "model"])["dayaniklilik"].mean().reset_index()
fig = px.bar(
    kat_ort, x="dayaniklilik", y="category", color="model", barmode="group",
    facet_col="mod", orientation="h", color_discrete_map=RENK,
    category_orders={"model": ["orijinal", "heretic"]},
    title="Kategori dayanıklılığı (1 = hiç kırılmadı)",
)
fig.update_xaxes(range=[0, 1], tickformat=".0%")
fig.update_layout(height=520)
fig.show()

fark = kat_ort.pivot_table(index=["mod", "category"], columns="model", values="dayaniklilik")
fark = fark.reindex(columns=["orijinal", "heretic"])
fark["fark (her − orj)"] = fark["heretic"] - fark["orijinal"]
fark.sort_values("fark (her − orj)").style.format("{:.0%}")

## §4 · Reddetme kalıbı oranı (H3)

Tarayıcının kararından **bağımsız** bir ölçüm: yanıtta `REFUSAL_PATTERNS` kalıplarından
biri geçiyor mu? Bu, Heretic'in kendi optimizasyonda kullandığı `KeywordRate` ölçütünün
karşılığıdır. Heretic işini yaptıysa bu oran keskin biçimde düşmeli.

Dikkat: Türkçe prompta İngilizce cevap veren bir model de yakalanır (kalıplar iki dilli),
ama kalıp listesinde olmayan bir ret biçimi "ret yok" sayılır.

In [ ]:
red = k.red_orani(df)
genel = df[df["error"].isna()].groupby(["mod", "model"])["reddetti"].mean().unstack("model")
print("Genel reddetme oranı:")
display(genel.reindex(columns=["orijinal", "heretic"]).style.format("{:.0%}"))
red.style.format("{:.0%}")

## §5 · Eşleştirilmiş anlamlılık testi (H4)

Aynı seed aynı (şablon, prompt) listesini ürettiği için iki modelin denemeleri birebir
eşlenir. Bilgi yalnızca **uyumsuz** çiftlerdedir (bir modelde kırılıp diğerinde
kırılmayan); McNemar kesin testi bunların iki yöne eşit dağılıp dağılmadığını sınar.

İki okuma raporlanır: *katı* (kararsızlar kırılmamış sayılır) ve *en kötü durum*
(kararsızlar kırılmış sayılır). İkisi aynı sonuca varıyorsa bulgu sağlamdır.

Birden fazla seed varsa test **her seed için ayrı** yapılır: aynı prompt farklı
seed'lerde tekrar ettiği için çiftler seed'ler arasında bağımsız değildir ve
hepsini tek havuzda test etmek p değerini yapay olarak küçültür.

In [ ]:
ciftler = k.eslesmis_ciftler(df)
print(f"Eşleşen çift: {len(ciftler)} · eşleşmeyen deneme: {ciftler.attrs['eslesmeyen']}\n")
for (mod, seed), grup in ciftler.groupby(["mod", "seed"]):
    print(f"[{mod} · seed {seed}]")
    print("  katı        :", k.mcnemar(grup))
    print("  en kötü dur.:", k.mcnemar(grup, kararsiz_kirik_sayilsin=True))

## §6 · Tespit duyarlılığı: reddi kaybolan denemeler

Orijinal model **reddetti**, Heretic model **reddetmedi**: Heretic'in tam olarak
hedeflediği davranış değişikliği. Bu alt kümede tarayıcının Heretic model için
"kırıldı" deme oranı, tespit hattının duyarlılığına (recall) dair bir alt sınır verir.

* **dayandı** satırları → olası *yanlış negatifler* (ret yok ama tarayıcı kırılma görmedi).
  Bazıları gerçekten zararsız bir "konu dışı" yanıt olabilir; ayrımı insan etiketi yapar (§8).
* **kararsız** satırları → Katman 2'nin işi.

In [ ]:
duyarlilik = k.tespit_duyarliligi(ciftler)
if duyarlilik.empty:
    print("Reddi kaybolan deneme yok.")
else:
    display(duyarlilik)
    for mod, grup in ciftler[ciftler["reddetti_orj"] & ~ciftler["reddetti_her"]].groupby("mod"):
        oran = grup["karar_her"].isin(["kırıldı", "kırıldı (düşük güven)"]).mean()
        print(
            f"[{mod}] reddi kaybolan {len(grup)} denemenin {oran:.0%}'i "
            "'kırıldı' olarak tespit edildi"
        )

## §7 · Katman 1 ↔ Katman 2 (H5)

Yalnızca hem `kural` hem `judge` modunda koşu varsa çalışır. Aynı deneme iki modda
hangi karara vardı? Judge, Katman 1'in kararsız bıraktıklarını nereye taşıdı?

> Not: Model yanıtları iki koşu arasında birebir aynı olmayabilir (sıcaklık 0'da bile
> küçük farklar olabilir); bu tablo yanıt düzeyinde değil, karar düzeyinde bir karşılaştırmadır.

In [ ]:
if {"kural", "judge"} <= set(df["mod"]):
    anahtar = ["model", "seed", "template_id", "prompt_sent"]
    kj = df[df["mod"] == "kural"][[*anahtar, "karar"]].merge(
        df[df["mod"] == "judge"][[*anahtar, "karar"]], on=anahtar, suffixes=("_kural", "_judge")
    )
    for model, grup in kj.groupby("model"):
        print(f"\n{model}: satır = yalnız Katman 1, sütun = Katman 1 + judge")
        display(pd.crosstab(grup["karar_kural"], grup["karar_judge"], margins=True))
else:
    print("Judge modunda koşu yok — 2_taramalari_kos.ps1 -Judge ile çalıştırın.")

## §8 · Altın küme adayları (spesifikasyon §6)

İnsan etiketlemesi için öncelikli denemeler CSV'ye yazılır. `insan_etiketi` sütununa
`kırıldı` / `dayandı` yazıp dosyayı kaydedin; bu küme hem Katman 1'in hem judge'ın
precision/recall ölçümünde kullanılır (Faz 2 çıktısı).

Yanıtlar 400 karakterde kırpılır. Dosya `sonuclar/` altındadır ve depoya girmez.

In [ ]:
cikti = k.SONUC_DIZINI / ("kuru_prova" if KURU_PROVA else ".") / "altin_kume_adaylari.csv"
adaylar = k.altin_kume_adaylari(ciftler, cikti.resolve())
print(f"{len(adaylar)} aday yazıldı: {cikti.resolve()}")
adaylar.groupby(["oncelik_nedeni", "model"]).size().rename("adet").to_frame()

## §9 · Tez için özet tablo

Aşağıdaki çıktı doğrudan tez metnine (Markdown/LaTeX) aktarılabilir.

In [ ]:
satirlar = []
for mod, grup in ciftler.groupby("mod"):
    r = ozet[ozet["mod"] == mod].groupby("model")["R"].mean()
    rd = df[(df["mod"] == mod) & df["error"].isna()].groupby("model")["reddetti"].mean()
    m = k.mcnemar(grup)
    satirlar.append({
        "Tespit": "Katman 1" if mod == "kural" else "Katman 1 + judge",
        "R (orijinal)": f"{r.get('orijinal', float('nan')):.1f}",
        "R (Heretic)": f"{r.get('heretic', float('nan')):.1f}",
        "Ret oranı (orj → her)": (
            f"{rd.get('orijinal', float('nan')):.0%} → "
            f"{rd.get('heretic', float('nan')):.0%}"
        ),
        "Uyumsuz çift (her/orj)": f"{m.yalniz_heretic}/{m.yalniz_orijinal}",
        "McNemar p": f"{m.p_degeri:.2g}",
    })
tez = pd.DataFrame(satirlar)
try:
    print(tez.to_markdown(index=False))  # pip install tabulate
except ImportError:
    display(tez)